# Reel Insights: Pandas Analysis

This notebook uses Pandas and NumPy to analyse the cleaned movie dataset.

We will explore movie statistics, profitability, return on investment (ROI), genres, actors, release years, and data quality.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

movies = pd.read_csv("../data/movies_cleaned_corrected.csv")
genres = pd.read_csv("../data/genres_cleaned.csv")
movie_genres = pd.read_csv("../data/movie_genres_cleaned.csv")
cast = pd.read_csv("../data/cast_cleaned.csv")
crew = pd.read_csv("../data/crew_cleaned.csv")
movie_keywords = pd.read_csv("../data/movie_keywords_cleaned.csv")

print("All six datasets loaded successfully!")

All six datasets loaded successfully!


In [4]:
# Question 1: List the total number of unique movies, genres,
# cast members and crew members collected across all tables.

print("QUESTION 1: UNIQUE RECORD COUNTS")
print("-" * 45)

print("Total unique movies:", movies["movie_id"].nunique())
print("Total unique genres:", genres["genre_id"].nunique())
print("Total unique cast members:", cast["person_id"].nunique())
print("Total unique crew members:", crew["person_id"].nunique())

QUESTION 1: UNIQUE RECORD COUNTS
---------------------------------------------
Total unique movies: 2503
Total unique genres: 19
Total unique cast members: 11300
Total unique crew members: 2848


### Insight

The dataset contains 2,503 unique movies and 19 unique genres, together with a large number of unique cast and crew members. This confirms that the dataset provides substantial coverage of movie, genre and talent information for further relational and exploratory analysis.

In [5]:
# Question 2: List all movies with missing or zero budget/revenue values,
# and decide how these should be treated for the rest of the analysis.

financial_issues = movies[
    movies["budget"].isna() |
    movies["revenue"].isna() |
    (movies["budget"] == 0) |
    (movies["revenue"] == 0)
].copy()

print("QUESTION 2: MISSING OR ZERO BUDGET/REVENUE")
print("-" * 55)

print("Movies with missing budget:",
      movies["budget"].isna().sum())

print("Movies with zero budget:",
      (movies["budget"] == 0).sum())

print("Movies with missing revenue:",
      movies["revenue"].isna().sum())

print("Movies with zero revenue:",
      (movies["revenue"] == 0).sum())

print("\nTotal movies with at least one budget/revenue issue:",
      len(financial_issues))

print("\nMOVIES WITH MISSING OR ZERO FINANCIAL VALUES")
print("-" * 55)

print(
    financial_issues[
        ["title", "budget", "revenue"]
    ].to_string(index=False)
)

QUESTION 2: MISSING OR ZERO BUDGET/REVENUE
-------------------------------------------------------
Movies with missing budget: 0
Movies with zero budget: 90
Movies with missing revenue: 0
Movies with zero revenue: 102

Total movies with at least one budget/revenue issue: 133

MOVIES WITH MISSING OR ZERO FINANCIAL VALUES
-------------------------------------------------------
                                                                                                                          title      budget     revenue
                                                                                                   Zack Snyder's Justice League  70000000.0         0.0
                                                                                                                       Bird Box  19800000.0         0.0
                                                                                                                       365 Days         0.0   9458590.0
              

### Insight

There are no missing budget or revenue values in the dataset; however, 90 movies have a budget recorded as zero and 102 movies have revenue recorded as zero. Overall, 133 movies have at least one zero financial value. Since TMDB uses zero to represent unknown financial information, these movies will be excluded from analyses requiring valid financial figures, such as profit, ROI and budget-versus-revenue comparisons. They will still be retained for non-financial analyses such as ratings, genres, popularity, runtime and release trends.

In [6]:
# Question 3: What are the profit and ROI values for movies
# with valid budget and revenue figures?

# Keep only movies with valid positive budget and revenue
valid_movies = movies[
    (movies["budget"] > 0) &
    (movies["revenue"] > 0)
].copy()

# Calculate profit
valid_movies["profit"] = (
    valid_movies["revenue"] - valid_movies["budget"]
)

# Calculate Return on Investment (ROI)
valid_movies["ROI"] = (
    valid_movies["profit"] / valid_movies["budget"]
) * 100

print("QUESTION 3: MOVIE PROFIT AND ROI")
print("-" * 50)

print(
    "Movies with valid budget and revenue:",
    len(valid_movies)
)

print("\nPROFIT AND ROI FOR ALL VALID MOVIES")
print("-" * 50)

print(
    valid_movies[
        ["title", "budget", "revenue", "profit", "ROI"]
    ].to_string(index=False)
)

QUESTION 3: MOVIE PROFIT AND ROI
--------------------------------------------------
Movies with valid budget and revenue: 2370

PROFIT AND ROI FOR ALL VALID MOVIES
--------------------------------------------------
                                                                              title       budget      revenue        profit           ROI
                                                                       Interstellar 1.650000e+08 7.466067e+08  5.816067e+08  3.524889e+02
                                                                          Inception 1.600000e+08 8.390306e+08  6.790306e+08  4.243941e+02
                                                                       The Avengers 2.200000e+08 1.518816e+09  1.298816e+09  5.903707e+02
                                                                    The Dark Knight 1.850000e+08 1.004558e+09  8.195584e+08  4.430046e+02
                                                                             Avatar 2.370000e+0

### Insight

A total of 2,370 movies have valid positive budget and revenue figures and were therefore suitable for profit and ROI analysis. The results show that profitability varies substantially across movies: a film can generate a large absolute profit while another film may achieve a stronger ROI because it earned more relative to its production budget. Therefore, both profit and ROI should be considered when evaluating financial performance rather than relying on either measure alone.

In [7]:
# Question 4: What are the top 15 movies by ROI
# with a minimum budget of $1 million?

# Keep movies with a budget of at least $1 million
roi_eligible_movies = valid_movies[
    valid_movies["budget"] >= 1_000_000
].copy()

# Sort by ROI from highest to lowest
top_15_roi = roi_eligible_movies.sort_values(
    by="ROI",
    ascending=False
).head(15)

print("QUESTION 4: TOP 15 MOVIES BY ROI")
print("Minimum budget: $1,000,000")
print("-" * 70)

print(
    top_15_roi[
        ["title", "budget", "revenue", "profit", "ROI"]
    ].to_string(index=False)
)

QUESTION 4: TOP 15 MOVIES BY ROI
Minimum budget: $1,000,000
----------------------------------------------------------------------
                          title     budget     revenue      profit          ROI
       Dragon Ball Super: Broly  1000000.0 125002821.0 124002821.0 12400.282100
Snow White and the Seven Dwarfs  1488423.0 184925486.0 183437063.0 12324.256142
  The Rocky Horror Picture Show  1400000.0 171181400.0 169781400.0 12127.242857
                          Rocky  1000000.0 117253345.0 116253345.0 11625.334500
             Gone with the Wind  4000000.0 402352579.0 398352579.0  9958.814475
                The Jungle Book  4000000.0 378000000.0 374000000.0  9350.000000
                     Cinderella  2900000.0 263600000.0 260700000.0  8989.655172
                            Saw  1200000.0 104045735.0 102845735.0  8570.477917
 One Hundred and One Dalmatians  3600000.0 303000000.0 299400000.0  8316.666667
     E.T. the Extra-Terrestrial 10500000.0 797307407.0 786807407.0  7

### Insight

Among movies with a production budget of at least $1 million, *Dragon Ball Super: Broly* achieved the highest ROI at approximately 12,400%, followed closely by *Snow White and the Seven Dwarfs* and *The Rocky Horror Picture Show*. The results show that several relatively low-budget films generated exceptionally high returns compared with their production costs. This demonstrates that a larger production budget does not necessarily lead to a higher return on investment, and financially successful films can achieve strong returns through relatively modest budgets.

In [8]:
# Question 5: How many movies were released each year,
# and which year had the highest movie output?

# Step 1: Convert release_date to datetime format
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
)

# Step 2: Extract the release year
movies["release_year"] = movies["release_date"].dt.year

# Step 3: Count the number of unique movies released each year
movies_per_year = (
    movies.dropna(subset=["release_year"])
    .groupby("release_year")["movie_id"]
    .nunique()
    .sort_index()
)

# Step 4: Identify the year with the highest movie output
highest_year = movies_per_year.idxmax()
highest_count = movies_per_year.max()

# Step 5: Display all yearly results
print("QUESTION 5: MOVIE RELEASES PER YEAR")
print("-" * 50)

print("\nNumber of movies released each year:")
print(movies_per_year.to_string())

print("\nYear with the highest movie output:")
print(
    int(highest_year),
    "-",
    highest_count,
    "movies"
)

QUESTION 5: MOVIE RELEASES PER YEAR
--------------------------------------------------

Number of movies released each year:
release_year
1921      1
1922      1
1927      1
1931      2
1936      1
1938      1
1939      2
1940      3
1941      2
1942      1
1943      1
1946      1
1948      2
1950      3
1951      1
1952      1
1953      2
1954      3
1955      1
1957      4
1958      1
1959      5
1960      3
1961      2
1962      4
1963      5
1964      4
1965      3
1966      2
1967      3
1968      5
1969      1
1970      1
1971      4
1972      1
1973      5
1974      5
1975      6
1976      5
1977      7
1978      5
1979     11
1980      8
1981      7
1982     11
1983      6
1984     16
1985     11
1986     13
1987     18
1988     17
1989     14
1990     18
1991     14
1992     20
1993     23
1994     24
1995     33
1996     23
1997     38
1998     42
1999     39
2000     49
2001     47
2002     53
2003     58
2004     73
2005     64
2006     62
2007     78
2008     81
2009     8

### Insight

The number of movies represented in the dataset generally increases over time, with a particularly noticeable rise from the 1990s onwards. The highest movie output occurs in 2016, with 121 movies in the dataset. The lower counts in the most recent years, especially 2025 and 2026, should be interpreted cautiously because the dataset may not contain complete coverage for these years.

In [9]:
# Question 6: Which movies have unusual runtimes?

print("QUESTION 6: MOVIE RUNTIME OUTLIERS")
print("-" * 50)

# Step 1: Display runtime summary statistics
print("\nRuntime summary:")
print(movies["runtime"].describe())

# Step 2: Calculate Q1 and Q3
Q1 = movies["runtime"].quantile(0.25)
Q3 = movies["runtime"].quantile(0.75)

# Step 3: Calculate the Interquartile Range (IQR)
IQR = Q3 - Q1

# Step 4: Calculate the outlier limits
lower_limit = Q1 - 1.5 * IQR
upper_limit = Q3 + 1.5 * IQR

print("\nLower outlier limit:", lower_limit, "minutes")
print("Upper outlier limit:", upper_limit, "minutes")

# Step 5: Identify runtime outliers
runtime_outliers = movies[
    (movies["runtime"] < lower_limit) |
    (movies["runtime"] > upper_limit)
].copy()

print("\nNumber of runtime outliers:",
      len(runtime_outliers))

# Step 6: Display the 10 longest runtime outliers
longest_outliers = runtime_outliers.sort_values(
    by="runtime",
    ascending=False
).head(10)

print("\n10 longest runtime outliers:")

print(
    longest_outliers[
        ["title", "runtime"]
    ].to_string(index=False)
)

QUESTION 6: MOVIE RUNTIME OUTLIERS
--------------------------------------------------

Runtime summary:
count    2503.000000
mean      114.413903
std        21.418867
min        45.000000
25%        99.000000
50%       111.000000
75%       126.000000
max       242.000000
Name: runtime, dtype: float64

Lower outlier limit: 58.5 minutes
Upper outlier limit: 166.5 minutes

Number of runtime outliers: 60

10 longest runtime outliers:
                                        title  runtime
                 Zack Snyder's Justice League      242
                           Gone with the Wind      238
                  Once Upon a Time in America      229
                           Lawrence of Arabia      228
                                      Ben-Hur      212
                                 The Irishman      209
                                Seven Samurai      207
                   Killers of the Flower Moon      206
                        The Godfather Part II      202
The Lord of the 

### Insight

Using the IQR method, movie runtimes below 58.5 minutes or above 166.5 minutes were classified as statistical outliers, resulting in 60 unusual-runtime movies. The longest runtime in the dataset is *Zack Snyder's Justice League* at 242 minutes, followed by *Gone with the Wind* at 238 minutes. These results show that while most movies are concentrated around the median runtime of 111 minutes, a relatively small group of films have substantially shorter or longer runtimes.

In [10]:
# Question 7: What is each movie's primary genre,
# and what is the average popularity by primary genre?

print("QUESTION 7: AVERAGE POPULARITY BY PRIMARY GENRE")
print("-" * 60)

# Step 1: Connect movies to their genre IDs
movie_genre_data = movie_genres.merge(
    genres,
    on="genre_id",
    how="left"
)

# Step 2: Add movie popularity
movie_genre_data = movie_genre_data.merge(
    movies[["movie_id", "popularity"]],
    on="movie_id",
    how="left"
)

# Step 3: Use the first listed genre for each movie
# The first genre listed is treated as the primary genre
primary_genres = movie_genre_data.drop_duplicates(
    subset=["movie_id"],
    keep="first"
).copy()

# Step 4: Calculate movie count and average popularity
genre_popularity = (
    primary_genres
    .dropna(subset=["genre_name", "popularity"])
    .groupby("genre_name")
    .agg(
        movie_count=("movie_id", "nunique"),
        average_popularity=("popularity", "mean")
    )
    .reset_index()
)

# Step 5: Sort from highest to lowest average popularity
genre_popularity = genre_popularity.sort_values(
    by="average_popularity",
    ascending=False
)

genre_popularity["average_popularity"] = (
    genre_popularity["average_popularity"].round(2)
)

# Step 6: Display the results
print("\nAverage popularity by primary genre:")
print(genre_popularity.to_string(index=False))

print("\nTop 10 genres by average popularity:")
print(genre_popularity.head(10).to_string(index=False))

QUESTION 7: AVERAGE POPULARITY BY PRIMARY GENRE
------------------------------------------------------------

Average popularity by primary genre:
     genre_name  movie_count  average_popularity
Science Fiction          119               21.26
          Music           10               18.32
      Animation          139               16.81
         Family           48               16.71
         Action          474               16.39
            War           23               15.47
         Horror          223               14.85
      Adventure          244               14.79
        Fantasy           93               13.02
        Mystery           29               12.43
        Western           10               12.32
       Thriller           97               11.29
          Drama          447               11.28
          Crime          110               10.74
        History           12               10.41
         Comedy          361                9.98
        Romance     

### Insight

Science Fiction has the highest average popularity among the primary genres, with an average popularity score of 21.26 across 119 movies. Music ranks second with an average popularity of 18.32, although it contains only 10 movies, so its result is based on a much smaller sample. Animation, Family and Action also show relatively high average popularity, suggesting that these genres attract strong audience interest within this dataset.

In [11]:
# Question 8: Who are the top 10 most prolific actors by movie count,
# and what is the average rating of their movies?

print("QUESTION 8: TOP 10 MOST PROLIFIC ACTORS")
print("-" * 60)

# Step 1: Connect cast records with movie details
actor_movies = cast.merge(
    movies[["movie_id", "title", "vote_average"]],
    on="movie_id",
    how="left"
)

# Step 2: Group movies by actor
actor_summary = (
    actor_movies
    .dropna(subset=["person_id", "actor_name"])
    .groupby(["person_id", "actor_name"])
    .agg(
        movie_count=("movie_id", "nunique"),
        average_movie_rating=("vote_average", "mean")
    )
    .reset_index()
)

# Step 3: Find the 10 actors with the most movie appearances
top_actors = actor_summary.sort_values(
    by=["movie_count", "actor_name"],
    ascending=[False, True]
).head(10).copy()

# Step 4: Round the average rating
top_actors["average_movie_rating"] = (
    top_actors["average_movie_rating"].round(2)
)

# Step 5: Display the results
print("\nTop 10 actors by number of movie appearances:")

print(
    top_actors[
        ["actor_name", "movie_count", "average_movie_rating"]
    ].to_string(index=False)
)

QUESTION 8: TOP 10 MOST PROLIFIC ACTORS
------------------------------------------------------------

Top 10 actors by number of movie appearances:
        actor_name  movie_count  average_movie_rating
 Samuel L. Jackson           39                  6.94
         Brad Pitt           38                  7.26
    Robert De Niro           38                  7.23
       Johnny Depp           37                  6.86
         Tom Hanks           33                  7.29
     Mark Wahlberg           32                  6.63
Scarlett Johansson           32                  7.06
      Willem Dafoe           32                  7.11
    Morgan Freeman           31                  6.96
        Tom Cruise           31                  7.04


### Insight

Samuel L. Jackson is the most prolific actor in the dataset, appearing in 39 movies, followed by Brad Pitt and Robert De Niro with 38 movies each. Among the top 10 most prolific actors, Tom Hanks has the highest average movie rating at 7.29, followed by Brad Pitt at 7.26 and Robert De Niro at 7.23. This shows that a high number of movie appearances does not necessarily correspond to the highest average movie rating.

In [12]:
# Question 9: Which movie titles appear in multiple release years,
# and could they represent remakes or different films with the same title?

print("QUESTION 9: MOVIE TITLES REUSED ACROSS YEARS")
print("-" * 60)

# Step 1: Make sure release dates are in date format
movies["release_date"] = pd.to_datetime(
    movies["release_date"],
    errors="coerce"
)

movies["release_year"] = movies["release_date"].dt.year

# Step 2: Group movies that share the same title
title_year_counts = (
    movies.dropna(subset=["title", "release_year"])
    .groupby("title")
    .agg(
        number_of_years=("release_year", "nunique"),
        release_years=(
            "release_year",
            lambda years: ", ".join(
                map(str, sorted(years.astype(int).unique()))
            )
        ),
        number_of_movies=("movie_id", "nunique")
    )
    .reset_index()
)

# Step 3: Keep titles appearing in more than one year
reused_titles = title_year_counts[
    title_year_counts["number_of_years"] > 1
].sort_values(
    by=["number_of_years", "title"],
    ascending=[False, True]
)

# Step 4: Display the results
print(
    "\nNumber of titles released in multiple years:",
    len(reused_titles)
)

print("\nTitles released in multiple years:")
print(reused_titles.to_string(index=False))

QUESTION 9: MOVIE TITLES REUSED ACROSS YEARS
------------------------------------------------------------

Number of titles released in multiple years: 49

Titles released in multiple years:
                                   title  number_of_years    release_years  number_of_movies
                               Halloween                3 1978, 2007, 2018                 3
                              Robin Hood                3 1973, 2010, 2018                 3
               A Nightmare on Elm Street                2       1984, 2010                 2
                                 Aladdin                2       1992, 2019                 2
                     Alice in Wonderland                2       1951, 2010                 2
                    Beauty and the Beast                2       1991, 2017                 2
                                  Carrie                2       1976, 2013                 2
                        Charlie's Angels                2       2

### Insight

A total of 49 movie titles appear in multiple release years, indicating that repeated titles in the dataset do not necessarily represent duplicate records. *Halloween* and *Robin Hood* each appear in three different release years, while titles such as *Aladdin*, *Dune*, *The Lion King* and *The Jungle Book* appear in two different years. Many of these cases are likely to represent remakes, reboots, adaptations or different films sharing the same title. Therefore, movie titles alone should not be used to identify duplicate records; the movie ID and release year should also be considered.

In [13]:
# Question 10: Which movies do not follow the general relationship
# between popularity and vote count?

print("QUESTION 10: POPULARITY AND VOTE COUNT ANOMALIES")
print("-" * 60)

# Step 1: Select the columns needed
popularity_votes = movies[
    ["title", "popularity", "vote_count", "vote_average"]
].copy()

# Step 2: Remove rows with missing values
popularity_votes = popularity_votes.dropna(
    subset=["popularity", "vote_count"]
)

# Step 3: Calculate the general correlation
popularity_correlation = popularity_votes[
    "popularity"
].corr(popularity_votes["vote_count"])

print(
    "\nCorrelation between popularity and vote count:",
    round(popularity_correlation, 3)
)

# Step 4: Calculate thresholds
high_popularity = popularity_votes["popularity"].quantile(0.75)
low_popularity = popularity_votes["popularity"].quantile(0.25)

high_votes = popularity_votes["vote_count"].quantile(0.75)
low_votes = popularity_votes["vote_count"].quantile(0.25)

print("\nThresholds used:")
print("High popularity:", round(high_popularity, 2))
print("Low popularity:", round(low_popularity, 2))
print("High vote count:", round(high_votes, 2))
print("Low vote count:", round(low_votes, 2))

# Step 5: Find movies with high popularity but low vote count
high_pop_low_votes = popularity_votes[
    (popularity_votes["popularity"] >= high_popularity) &
    (popularity_votes["vote_count"] <= low_votes)
].sort_values(
    by="popularity",
    ascending=False
)

# Step 6: Find movies with high vote count but low popularity
high_votes_low_pop = popularity_votes[
    (popularity_votes["vote_count"] >= high_votes) &
    (popularity_votes["popularity"] <= low_popularity)
].sort_values(
    by="vote_count",
    ascending=False
)

print("\nHIGH POPULARITY BUT LOW VOTE COUNT")
print("-" * 50)

print(
    high_pop_low_votes[
        ["title", "popularity", "vote_count", "vote_average"]
    ].to_string(index=False)
)

print("\nHIGH VOTE COUNT BUT LOW POPULARITY")
print("-" * 50)

print(
    high_votes_low_pop[
        ["title", "popularity", "vote_count", "vote_average"]
    ].to_string(index=False)
)

QUESTION 10: POPULARITY AND VOTE COUNT ANOMALIES
------------------------------------------------------------

Correlation between popularity and vote count: 0.361

Thresholds used:
High popularity: 14.87
Low popularity: 7.45
High vote count: 7276.0
Low vote count: 3027.0

HIGH POPULARITY BUT LOW VOTE COUNT
--------------------------------------------------
                                    title  popularity  vote_count  vote_average
                           Disclosure Day    394.3346        2328         7.427
                                Backrooms    258.4743        2483         7.100
                         Mortal Kombat II     99.5231        2260         7.906
                   Lee Cronin's The Mummy     79.4601        2430         7.943
                            The Housemaid     41.4337        2789         7.269
                 How to Train Your Dragon     35.4872        3008         7.929
             Final Destination Bloodlines     34.1109        3001         7.000


### Insight

There is a moderate positive correlation of 0.361 between movie popularity and vote count, indicating that movies with higher vote counts generally tend to be more popular, although the relationship is not strong. Several movies do not follow this general pattern. For example, *Disclosure Day* has very high popularity (394.33) but only 2,328 votes, while *The Silence of the Lambs* has 18,235 votes despite a relatively low popularity score of 7.06. These anomalies show that current popularity and accumulated vote count measure different aspects of audience engagement and do not always move together.

## Additional NumPy Analysis

The following analysis uses NumPy to compare individual movie ratings with the overall average rating and classify movies into above-average and below-average rating categories.

In [3]:
# NumPy Analysis: Compare movie ratings with the overall average

# Calculate the overall average movie rating using NumPy
average_rating = np.mean(
    movies["vote_average"].dropna().to_numpy()
)

# Use NumPy to classify movies as above or below the average rating
movies["rating_category"] = np.where(
    movies["vote_average"] >= average_rating,
    "Above Average",
    "Below Average"
)

print("NUMPY ANALYSIS: MOVIE RATING CLASSIFICATION")
print("-" * 50)

print(
    "Overall average movie rating:",
    round(average_rating, 2)
)

print("\nNumber of movies in each rating category:")
print(
    movies["rating_category"]
    .value_counts()
    .to_string()
)

NUMPY ANALYSIS: MOVIE RATING CLASSIFICATION
--------------------------------------------------
Overall average movie rating: 6.94

Number of movies in each rating category:
rating_category
Above Average    1268
Below Average    1235


### Insight

The overall average movie rating is 6.94. Using NumPy, 1,268 movies were classified as having an above-average rating, while 1,235 movies were classified as below average. The nearly even distribution indicates that movie ratings in the dataset are relatively balanced around the overall mean.

## Pandas and NumPy Analysis Summary

The cleaned TMDB datasets were analysed using Pandas and NumPy to explore movie performance, profitability, release trends, genres, actors, runtime patterns and audience engagement.

The analysis demonstrated:

- Data inspection and quality assessment using Pandas.
- Identification and appropriate treatment of zero or invalid financial values.
- Feature engineering through the calculation of movie profit, ROI, release year and rating category.
- Merging related datasets using movie, genre and person identifiers.
- Grouping and aggregation to analyse genres and actor performance.
- Analysis of annual movie release trends and identification of the highest-output year.
- Runtime outlier detection using the Interquartile Range (IQR) method.
- Identification of repeated movie titles across different release years to distinguish potential remakes, reboots or different films sharing the same title.
- Analysis of the relationship between movie popularity and vote count, including identification of movies that do not follow the general pattern.
- NumPy operations using `np.mean()` to calculate the overall average rating and `np.where()` to classify movies as above or below the dataset average.

The analysis found an overall average movie rating of approximately 6.94. Using NumPy-based classification, 1,268 movies were classified as having an above-average rating and 1,235 as having a below-average rating.

The results also identified important patterns in financial performance, genre popularity, actor activity, movie release trends, runtime outliers and audience engagement. These analyses demonstrate how Pandas and NumPy can be used for data manipulation, feature engineering, statistical exploration and insight generation from relational movie data.